---
## Question 4: Open-Shell Systems (Unrestricted Hartree-Fock)

So far, we have exclusively used **Restricted Hartree-Fock (RHF)**. RHF assumes our system is a "closed shell," meaning every spatial orbital contains exactly two electrons—one spin-up ($\alpha$) and one spin-down ($\beta$). Because they share the exact same spatial cloud, we only needed one Density Matrix ($\mathbf{P}$) and one Fock Matrix ($\mathbf{F}$).

But what if we want to simulate a radical, or a **Lithium** atom ($Z=3$)? 
Lithium has 3 electrons: two $\alpha$ electrons and one $\beta$ electron. 

Because there are more $\alpha$ electrons than $\beta$ electrons, the "Mean-Field" they generate is entirely different! The $\alpha$ electrons experience more quantum Exchange stabilization ($K$) than the lone $\beta$ electron. Consequently, the $\alpha$ electrons will spatially relax and change their shape, meaning the $\alpha$ cloud and the $\beta$ cloud will no longer perfectly overlap. This physical effect is called **Spin Polarization**.

To solve this, we must upgrade to **Unrestricted Hartree-Fock (UHF)**. We must split our engine into two parallel tracks: one for the $\alpha$ electrons and one for the $\beta$ electrons.



### The Mathematics of UHF (The Pople-Nesbet Equations)

In UHF, we have two separate Density Matrices ($\mathbf{P}^\alpha$ and $\mathbf{P}^\beta$) and two separate Fock Matrices ($\mathbf{F}^\alpha$ and $\mathbf{F}^\beta$). 

The One-Electron terms ($\mathbf{h}_{core}$) are exactly the same for both. But the Mean-Field repulsion completely changes:

1. **The Coulomb Term (Classical Repulsion):**
Classical electrostatics does not care about quantum spin. An electron feels the repulsive push from *all* other electrons in the system. Therefore, the Coulomb integral uses the **Total Density**: 
$$\mathbf{P}^{tot} = \mathbf{P}^\alpha + \mathbf{P}^\beta$$

2. **The Exchange Term (Quantum Glue):**
As we proved in the theoretical derivation, the Exchange Integral ($K$) goes to strictly zero for electrons with opposite spins. Therefore, an $\alpha$ electron only exchanges with other $\alpha$ electrons! The exchange term must only use the specific spin-density.

**The UHF Fock Equations:**
$$F^\alpha_{rs} = h_{rs} + \sum_{t} \sum_{u} \left[ P^{tot}_{tu} (rs\vert{}tu) - P^\alpha_{tu} (ru\vert{}ts) \right]$$
$$F^\beta_{rs} = h_{rs} + \sum_{t} \sum_{u} \left[ P^{tot}_{tu} (rs\vert{}tu) - P^\beta_{tu} (ru\vert{}ts) \right]$$

**Your Task:**
Below is the skeleton for a UHF SCF engine simulating the **Lithium** atom. 
1. Build the $\mathbf{F}^\alpha$ and $\mathbf{F}^\beta$ matrices using the equations above.
2. Diagonalize both of them independently.
3. Rebuild $\mathbf{P}^\alpha$ (using $N_{occ}^\alpha = 2$) and $\mathbf{P}^\beta$ (using $N_{occ}^\beta = 1$).
4. Calculate the total energy!



In [ ]:
# --- HELPER FUNCTION (DO NOT MODIFY) ---
def get_lithium_matrices():
    """Returns mock 3x3 integral matrices simulating a Lithium atom."""
    N_basis = 3
    S = np.eye(N_basis) + 0.1 * np.ones((N_basis, N_basis))
    np.fill_diagonal(S, 1.0)
    h = np.diag([-4.0, -1.5, -0.5]) 
    V = np.zeros((N_basis, N_basis, N_basis, N_basis))
    for i in range(N_basis):
        for j in range(N_basis):
            V[i,i,j,j] = 0.9; V[i,j,i,j] = 0.3
    return S, h, V
# ---------------------------------------

S_li, h_li, V_ee = get_lithium_matrices()
N = len(S_li)
X_li = fractional_matrix_power(S_li, -0.5)

# Lithium has 3 electrons. 2 spin-up, 1 spin-down.
N_alpha = 2
N_beta = 1

# Initial Guesses (Zeros)
P_alpha = np.zeros((N, N))
P_beta = np.zeros((N, N))
energy_old = 0.0

print("Iter | Total Electronic Energy | Delta E")
print("------------------------------------------")

for iteration in range(25):
    # 1. Total Density
    P_tot = P_alpha + P_beta
    
    # 2. Build Fock Matrices
    F_alpha = np.zeros((N, N))
    F_beta = np.zeros((N, N))
    
    for r in range(N):
        for s in range(N):
            F_alpha[r, s] = h_li[r, s]
            F_beta[r, s] = h_li[r, s]
            for t in range(N):
                for u in range(N):
                    # YOUR CODE HERE: Add the Coulomb and Exchange terms!
                    # Hint: Coulomb uses P_tot, Exchange uses P_alpha or P_beta
                    
                    # F_alpha[r, s] += ...
                    # F_beta[r, s] += ...
                    pass
                    
    # 3. Transform and Solve for Alpha
    F_alpha_prime = np.dot(X_li, np.dot(F_alpha, X_li))
    E_alpha, C_alpha_prime = la.eigh(F_alpha_prime)
    C_alpha = np.dot(X_li, C_alpha_prime)
    
    # 4. Transform and Solve for Beta
    F_beta_prime = np.dot(X_li, np.dot(F_beta, X_li))
    E_beta, C_beta_prime = la.eigh(F_beta_prime)
    C_beta = np.dot(X_li, C_beta_prime)
    
    # 5. Rebuild Density Matrices
    P_alpha_new = np.zeros((N, N))
    P_beta_new = np.zeros((N, N))
    
    for t in range(N):
        for u in range(N):
            # YOUR CODE HERE: Build P_alpha_new (Looping over N_alpha)
            # YOUR CODE HERE: Build P_beta_new (Looping over N_beta)
            # Note: Do NOT multiply by 2 anymore! Each spatial orbital only holds 1 specific spin.
            pass
            
    # 6. Calculate Total UHF Energy
    # Formula: E = 0.5 * Sum[ P_alpha*(h + F_alpha) + P_beta*(h + F_beta) ]
    E_elec = 0.5 * np.sum(P_alpha_new * (h_li + F_alpha) + P_beta_new * (h_li + F_beta))
    
    delta_E = abs(E_elec - energy_old)
    print(f"{iteration+1:4d} | {E_elec:23.6f} | {delta_E:.2e}")
    
    if delta_E < 1e-6: break
    P_alpha, P_beta = P_alpha_new, P_beta_new
    energy_old = E_elec

print("\nUHF SCF Converged!")



---
### Conceptual Checkpoint: The Danger of UHF

**Question 4.1:** In the RHF code, when building the Density Matrix, we multiplied the coefficients by $2$ (e.g., `P[t,u] = 2 * C[t,0] * C[u,0]`). Notice in the UHF code above, you were instructed *not* to multiply by 2. Based on the physics of Spin-Orbitals and the Pauli Exclusion Principle, why did that $2$ disappear?

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**

**Question 4.2 (Spin Contamination):** By giving the $\alpha$ and $\beta$ electrons complete freedom to occupy different spatial shapes, we gain a much more accurate energy for radicals. However, there is a dangerous side effect. 

Because the $\alpha$ and $\beta$ clouds no longer perfectly match, the total wavefunction is no longer a pure "Doublet" spin state (where Total Spin $S = \frac{1}{2}$). The math accidentally mixes in higher energy spin states (like Quartets, where $S = \frac{3}{2}$). This mathematical pollution is called **Spin Contamination**. 

If you were a computational chemist analyzing a highly spin-contaminated output file for a drug molecule, would you trust the optimized geometry? Why or why not?

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**

